# ЛЛ1 — запуск в Google Colab

Выберите **среду Python 3, CPU**, затем **«Среда выполнения → Выполнить все»**. Начальная ячейка получает проект с GitHub. По умолчанию блокнот анализирует уже выполненный эксперимент из 360 ответов без загрузки моделей.

Для нового эксперимента используйте последнюю ячейку. Все пояснения на русском. Если репозиторий закрыт, загрузите архив `LL1_masterat_ru.zip` через панель «Файлы» перед запуском первой ячейки.

In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys
import tempfile
import zipfile

REPOSITORY = "https://github.com/alinaRN1/llm1.git"
ROOT = Path(os.environ.get("LL1_PROJECT_ROOT", "/content/llm1"))
archives = [Path("/content/LL1_masterat_ru.zip"), Path("/content/LL1_masterat.zip")]
archive = next((p for p in archives if p.is_file()), None)

if not (ROOT / "data/questions.json").is_file():
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    if archive is not None:
        destination = Path(tempfile.mkdtemp(prefix="ll1_upload_", dir=ROOT.parent))
        with zipfile.ZipFile(archive) as bundle:
            for entry in bundle.infolist():
                target = (destination / entry.filename).resolve()
                if not target.is_relative_to(destination.resolve()):
                    raise ValueError("Архив содержит путь за пределами папки распаковки")
            bundle.extractall(destination)
        candidates = [destination, *[p for p in destination.iterdir() if p.is_dir()]]
        matches = [p for p in candidates if (p / "data/questions.json").is_file()]
        if len(matches) != 1:
            raise RuntimeError("В архиве не удалось однозначно найти проект лабораторной")
        ROOT = matches[0]
    else:
        if ROOT.exists() and any(ROOT.iterdir()):
            raise RuntimeError("Папка проекта занята другими файлами; выберите новую LL1_PROJECT_ROOT")
        result = subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(ROOT)],
                                capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError("Не удалось скачать репозиторий. Если он закрыт, загрузите "
                               "LL1_masterat_ru.zip в панель «Файлы» Colab и повторите ячейку.")

os.environ["LL1_PROJECT_ROOT"] = str(ROOT.resolve())
os.chdir(ROOT)
packages = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
print("Проект подготовлен:", ROOT)
print("Просмотр сохранённых результатов готов; скачивание весов не требуется.")


# ЛЛ1 — оценка языковых моделей с открытыми весами

Исследование магистерского уровня. Здесь анализируются результаты собственного локального запуска: **360 ответов**, три модели, два режима и три начальных значения. Синтетические данные программных тестов в эти результаты не входят. Все пояснения — на русском; исходный эксперимент проведён на румынском.

In [ ]:
from pathlib import Path
import json, os, sys, subprocess
import pandas as pd
from IPython.display import display, Image
ROOT = Path(os.environ["LL1_PROJECT_ROOT"])
RUN = ROOT / "results/final"
assert (ROOT / "data/questions.json").exists(), "Откройте блокнот из корня проекта"
sys.path.insert(0, str(ROOT / "src"))
from presentation import localize_records
pd.set_option("display.max_colwidth", 100)
def show_ru(records):
    display(pd.DataFrame(localize_records(records)))
print("Версия Python:", sys.version.split()[0])

## Набор оценки

Эталоны заданы до запуска и не передавались моделям. Для чтения ниже показан вспомогательный русский перевод; входами моделей были румынские оригиналы из `data/questions.json`.

In [ ]:
questions = json.loads((ROOT / "data/questions_ru.json").read_text())
show_ru([{k:q[k] for k in ["id", "category", "question", "reference_answer"]} for q in questions])

## Происхождение моделей

GGUF F16 из Docker AI. Каждый манифест и файл проверен SHA-256. Регистр тега важен: SmolLM2 `360m-f16` содержит Instruct.

In [ ]:
specs = json.loads((ROOT / "data/models_oci.lock.json").read_text())
show_ru([{
    "model": m["model"], "manifest_digest": m["manifest_digest"],
    "GGUF_bytes": next(x["size"] for x in m["layers"] if x["file"] == "model.gguf")
} for m in specs])

## Полнота и целостность данных

In [ ]:
import hashlib
manifest = json.loads((RUN / "manifest.json").read_text())
rows = json.loads((RUN / "scored.json").read_text())
assert manifest["status"] == "complete" and len(rows) == 360
assert len({(r["model"], r["id"], r["mode"], r["repetition"]) for r in rows}) == 360
assert all("audit" in r for r in rows)
assert manifest["dataset_sha256"] == hashlib.sha256((ROOT / "data/questions.json").read_bytes()).hexdigest()
assert all(r["answer_sha256"] == hashlib.sha256(r["answer"].encode()).hexdigest() for r in rows)
print("Все 360 ответов проверены; хеши набора и текстов совпадают.")
print("Процессор:", manifest["cpu"], "| версия реализации:", manifest["backend_version"])

## Строгая оценка и проверка содержания

Правильный ответ может не содержать маркер. Нарушение формата рассматривается отдельно от фактической ошибки.

In [ ]:
aggregate = pd.read_csv(RUN / "aggregate.csv")
show_ru(aggregate[["model", "mode", "factual_exact", "factual_content", "math_accuracy", "math_content", "steps_valid_rate", "complete_solution_rate"]].to_dict("records"))

In [ ]:
display(Image(filename=str(RUN / "audited_accuracy.png"), width=1100))

## Задержка, скорость и перплексия

Среда: CPU, 4 потока, F16. Низкая PPL не доказывает правильность. Референсная PPL считает все тексты одной Qwen и имеет стилевое смещение.

In [ ]:
show_ru(aggregate[["model", "mode", "ttft_median_ms", "latency_median_ms", "latency_p95_ms", "decode_tps_median", "reference_ppl_token_weighted", "truncation_rate"]].to_dict("records"))
display(Image(filename=str(RUN / "comparison.png"), width=1100))

In [ ]:
resources = json.loads((RUN / "resources.json").read_text())
show_ru([{
    "model": r["model"], "weights_GiB": r["weight_bytes"] / 2**30,
    "peak_RSS_GiB": r["sampled_peak_rss_bytes"] / 2**30,
    "native_PPL": r["corpus_native"]["ppl"],
    "native_bits_per_byte": r["corpus_native"]["bits_per_byte"]
} for r in resources])

## Неопределённость и парные сравнения

Единица перевыборки — вопрос; его повторы остаются вместе. Интервалы Уилсона используют первый повтор с n=12/8. Бутстреп разведочный.

In [ ]:
show_ru(json.loads((RUN / "content_comparisons.json").read_text()))
show_ru(json.loads((RUN / "content_decoding_comparisons.json").read_text()))

## Проверка математических решений

Верный результат ≠ верные шаги ≠ полное решение. Проверка одна, выполнена ИИ-ассистентом без независимого оценщика-человека. Решения связаны с хешами текстов.

In [ ]:
review = pd.read_csv(RUN / "manual_review.csv")
assert len(review) == 144
show_ru(review[["model", "mode", "id", "repetition", "content_correct", "steps_valid", "solution_complete", "note"]].head(16).to_dict("records"))

## Обсуждение и ограничения

In [ ]:
from IPython.display import Markdown
display(Markdown((RUN / "discussion.md").read_text()))

## Сохранённый протокол проверки программы

Ниже читается протокол тестов при подготовке работы. Это не новый запуск всех тестов в Colab. Тесты проверяют оценщик и обработку данных, а не качество языковых моделей.

In [ ]:
import xml.etree.ElementTree as ET
suite = ET.parse(ROOT / "reports/tests.xml").find("testsuite")
assert int(suite.attrib["failures"]) == 0 and int(suite.attrib["errors"]) == 0
print("В исходном протоколе пройдено тестов:", suite.attrib["tests"])
assert (ROOT / "reports/raport.pdf").exists()
print("Русский отчёт PDF/Word и приложения сохранены в проекте.")

## Новый эксперимент в Colab — по необходимости

Установите `RUN_NEW_EXPERIMENT = True` и выполните ячейку. Она установит CPU-реализацию llama.cpp, скачает около **4,2 ГБ весов**, получит **360 новых ответов** и скачает архив результатов. Сборка и генерация могут занять значительное время; оставьте сессию активной. Используется CPU с двумя потоками. Времена нового запуска относятся к аппаратуре Colab.

Сохранённый отчёт остаётся отчётом исходного эксперимента. Новые ответы нуждаются в отдельной проверке содержания и шагов: предыдущие решения проверяющего допустимы только при совпадении модели, задания и хеша ответа. Локальные файлы Colab удаляются при сбросе сессии, поэтому сохраните архив.

In [ ]:
RUN_NEW_EXPERIMENT = False

if not RUN_NEW_EXPERIMENT:
    print("Показаны сохранённые результаты. Для нового запуска включите флаг в этой ячейке.")
else:
    from datetime import datetime, timezone
    import shutil

    # Эта версия использует CPU, даже если в настройках Colab выбран GPU.
    environment = os.environ.copy()
    environment.update(CC="gcc", CXX="g++", CMAKE_BUILD_PARALLEL_LEVEL="2")
    subprocess.run([sys.executable, "-m", "pip", "install", "llama-cpp-python==0.3.16",
                    "psutil", "requests"], env=environment, check=True)
    subprocess.run([sys.executable, "src/pull_oci.py", "--locked", "data/models_oci.lock.json"],
                   check=True)
    RUN_ID = "colab-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    THREADS = min(2, os.cpu_count() or 1)
    subprocess.run([sys.executable, "src/benchmark_gguf.py", "--run-id", RUN_ID,
                    "--threads", str(THREADS)], check=True)
    NEW_RUN = ROOT / "results/runs" / RUN_ID
    subprocess.run([sys.executable, "src/analyze.py", str(NEW_RUN)], check=True)
    print("Новый эксперимент завершён:", NEW_RUN)
    print("Это новые измерения Colab. Проверку содержания и математических шагов нужно выполнить заново.")
    new_aggregate = pd.read_csv(NEW_RUN / "aggregate.csv")
    show_ru(new_aggregate[["model", "mode", "factual_exact", "math_accuracy",
                           "latency_median_ms", "decode_tps_median"]].to_dict("records"))
    display(Image(filename=str(NEW_RUN / "comparison.png"), width=1100))
    result_archive = shutil.make_archive(str(ROOT.parent / RUN_ID), "zip", root_dir=NEW_RUN)
    print("Архив новых результатов без весов:", result_archive)
    try:
        from google.colab import files
    except ImportError:
        pass
    else:
        files.download(result_archive)
